In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('AviationData.csv', encoding='latin1', low_memory=False)

print("Data Shape:", df.shape)
print("\n--- Missing Values ---")
print(df.isna().sum())

print("\n--- Summary Statistics ---")
df.describe(include='all')

In [ ]:
# Remove events older than 1983 (40-year threshold)
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df = df.dropna(subset=['Event.Date'])
df = df[df['Event.Date'].dt.year >= 1983]

# Filter strictly for professional builds and airplanes
df['Aircraft.Category'] = df['Aircraft.Category'].fillna('Airplane')
df = df[df['Aircraft.Category'].str.lower() == 'airplane']

df = df[df['Amateur.Built'].str.lower() == 'no']

print("Filtered Records Count:", len(df))

In [ ]:
# Fill missing injury counts with 0
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
df[injury_cols] = df[injury_cols].fillna(0)

# Estimate total passengers and drop zero-passenger entries
df['Total.Passengers'] = df[injury_cols].sum(axis=1)
df = df[df['Total.Passengers'] > 0]

# Calculate fatal/serious injury rate
df['Fatal_Serious_Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers']

# Aircraft destruction flag
df['Aircraft.damage'] = df['Aircraft.damage'].fillna('Unknown')
df['is_destroyed'] = (df['Aircraft.damage'].str.lower() == 'destroyed').astype(int)

In [ ]:
# Standardize Make and filter low-frequency makes (< 50 records)
df['Make'] = df['Make'].str.upper().str.strip()
make_counts = df['Make'].value_counts()
df = df[df['Make'].isin(make_counts[make_counts >= 50].index)]

# Standardize Model and construct unique plane identifier
df = df.dropna(subset=['Model'])
df['Model'] = df['Model'].str.upper().str.strip()
df['Plane_Type'] = df['Make'] + '_' + df['Model']

In [ ]:
# Clean categorical variables
df['Weather.Condition'] = df['Weather.Condition'].str.upper().replace({'UNK': 'UNKNOWN'}).fillna('UNKNOWN')
df['Engine.Type'] = df['Engine.Type'].str.title().replace({'Unknown': 'Other', 'None': 'Other'}).fillna('Other')

df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce')
df = df[(df['Number.of.Engines'] >= 1) & (df['Number.of.Engines'] <= 4)]

df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.title().fillna('Unknown')
df['Purpose.of.flight'] = df['Purpose.of.flight'].str.title().fillna('Unknown')

# Keep columns with at least 20,000 non-null values
df = df.dropna(thresh=20000, axis=1)

# Save cleaned dataset
df.to_csv('cleaned_aviation_data.csv', index=False)
print("Saved cleaned_aviation_data.csv successfully!")